# HASTIKA @ ICON-2026 — Test Phase: Task A + Task B
### Kannada-English Code-Mixed Comments

This notebook trains and submits for **both** sub-tasks now that the test phase has started:

- **Task A** — Binary: Hate / Non-Hate
- **Task B** — Multi-class (6-way): Gender, Political, Religion, Geo-political, Violence, Others

```
Upload+Inspect -> Fix mojibake -> Clean/Normalize -> Combine train+validation (both now labeled)
   -> TASK A: TF-IDF(word+char)+SVM  +  MuRIL transformer  -> tune ensemble on held-out split
            -> refit both on ALL labeled Task A data -> predict on hastika_binary_test.csv
   -> TASK B: TF-IDF(word+char)+SVM  +  MuRIL transformer  -> tune ensemble on held-out split
            -> refit both on ALL labeled Task B data -> predict on hastika_multiclass_test.csv
   -> two predictions.zip files, one per task
```

> ### ⚠️ Important discovery: the Kannada text in every CSV is double-encoded ("mojibake")
> Every file (`binary_train.csv`, the new validation files, the new test files, `multiclass_train.csv` —
> all of them) stores Kannada-script text **incorrectly double-encoded**: the original UTF-8 bytes were
> read as Latin-1/Windows-1252 and then re-saved as UTF-8. Example, from `binary_train.csv` row 2:
> the raw text `à²\x95à³\x8aà²¹à³\x8dà²²à²¿` is actually the Kannada word for "Kohli" (ಕೊಹ್ಲಿ), completely
> scrambled. Checking the whole dataset confirms **100% of Kannada-script content** is affected this way —
> there are zero rows with correctly-encoded Kannada anywhere in these files.
>
> This matters a lot: the earlier notebook's cleaning step kept only proper Kannada Unicode characters
> (`\u0C80-\u0CFF`) plus Latin letters — which means **every Kannada word in every comment was being
> silently deleted** by the character filter, since the mojibake bytes don't look like real Kannada to
> that filter. Fixed below using `ftfy.fix_text()`, which auto-detects and repairs this specific kind of
> encoding corruption (and safely leaves already-clean ASCII text untouched).
>
> This is likely a bigger lever than any of the ensembling/tuning done so far, since it recovers real
> content instead of squeezing more out of what was already visible to the model.

**Data used (upload all six CSVs into ONE Kaggle dataset):**
| File | Rows | Columns | Use |
|---|---|---|---|
| `binary_train.csv` | 6,446 | id, Comment, Label | Task A training data |
| `hastika_binary_validation.csv` | 806 | id, Comment, Label | Task A — **now labeled**, folded into training |
| `hastika_binary_test.csv` | 806 | id, Comment | Task A — **predict & submit this** |
| `multiclass_train.csv` | 3,159 | id, Comment, Hate Category | Task B training data |
| `hastika_multiclass_validation.csv` | 395 | id, Comment, Hate Category | Task B — **now labeled**, folded into training |
| `hastika_multiclass_test.csv` | 396 | id, Comment | Task B — **predict & submit this** |

**How to upload to Kaggle so the notebook finds the files easily:**
1. Put all six CSVs in **one single folder** on your computer, named e.g. `hastika-data`.
2. On Kaggle: **Datasets → New Dataset → Upload** that folder → give it a slug name (e.g. `hastika-icon-2026-data`) → **Create**.
3. In this notebook's right sidebar, **Add Input** → attach that dataset.
4. Run the **"Inspect dataset folder"** cell below — it prints the exact path Kaggle mounted your files at (usually `/kaggle/input/<your-dataset-slug>/`). Copy that path into `DATA_DIR` in the CONFIG cell if it doesn't already match.
5. Settings → Accelerator → **GPU T4 x2** (needed for the transformer cells).
6. Run all cells top to bottom. The notebook ends with two files: `predictions_taskA.zip` and `predictions_taskB.zip` in `/kaggle/working/` — download each and upload to the matching task's phase on CodaBench: https://www.codabench.org/competitions/17784/

In [ ]:
import os

# EDIT THIS to match your uploaded Kaggle dataset folder (see "Inspect dataset folder" cell below)
DATA_DIR = "/kaggle/input/hastika-icon-2026-data"

BIN_TRAIN_PATH   = os.path.join(DATA_DIR, "binary_train.csv")
BIN_VAL_PATH     = os.path.join(DATA_DIR, "hastika_binary_validation.csv")
BIN_TEST_PATH    = os.path.join(DATA_DIR, "hastika_binary_test.csv")

MC_TRAIN_PATH    = os.path.join(DATA_DIR, "multiclass_train.csv")
MC_VAL_PATH      = os.path.join(DATA_DIR, "hastika_multiclass_validation.csv")
MC_TEST_PATH     = os.path.join(DATA_DIR, "hastika_multiclass_test.csv")

OUTPUT_DIR = "/kaggle/working"
MODEL_NAME = "google/muril-base-cased"
MAX_LEN = 128
SEED = 42

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
!pip install -q -U "pandas<3.0" "numpy<2.1" "transformers>=4.40" datasets accelerate evaluate scikit-learn matplotlib seaborn
!pip install -q -U emoji ftfy
# IMPORTANT: if you already ran this cell once this session, use Kernel -> Restart Session
# (not just re-run this cell) before running everything else, or an old cached transformers
# import can cause "unexpected keyword argument" errors in TrainingArguments later.

In [ ]:
import re
import json
import random
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ftfy
import emoji
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: No GPU detected. Settings -> Accelerator -> GPU, then Restart & Run All.")

## Step 1 — Fix the double-encoded ("mojibake") Kannada text

Quick before/after on a known-bad row, then this fix is applied to every text column when each
CSV is loaded below.

In [ ]:
_demo_raw = "amar sir , à²\x95à³\x8aà²¹à³\x8dà²²à²¿ Bad performance, eshto world best players à²¨ Best performance."
print("BEFORE:", _demo_raw)
print("AFTER :", ftfy.fix_text(_demo_raw))

def fix_mojibake(text):
    return ftfy.fix_text(str(text))

## Step 2 — Shared text cleaning (used by both Task A and Task B)

In [ ]:
EMOTICON_MAP = {
    ":)": " smile ", ":-)": " smile ", ":(": " sad ", ":-(": " sad ",
    ":D": " laugh ", ":-D": " laugh ", ";)": " wink ", ";-)": " wink ",
    ":P": " tongue ", ":-P": " tongue ", "<3": " heart ", "</3": " heartbreak ",
    ":'(": " crying ", "XD": " laugh ", "xD": " laugh ",
}

def normalize_emoticons(text):
    for pat, repl in EMOTICON_MAP.items():
        text = text.replace(pat, repl)
    return text

KANNADA_RANGE = r"\u0C80-\u0CFF"

def clean_text(text):
    text = fix_mojibake(text)          # FIX: repair double-encoded Kannada/emoji FIRST
    text = text.strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)         # URLs
    text = re.sub(r"@\w+", " ", text)                       # mentions
    text = re.sub(r"&\w+;", " ", text)                      # HTML entities
    text = re.sub(r"<.*?>", " ", text)                       # HTML tags

    text = normalize_emoticons(text)
    text = emoji.demojize(text, delimiters=(" ", " "))       # emoji -> words, before char filter

    # keep Kannada script, English letters, digits, underscores (from demojize), basic punctuation
    text = re.sub(rf"[^{KANNADA_RANGE}a-zA-Z0-9_\s.,!?']", " ", text)
    text = re.sub(r"([.,!?])\1+", r"\1", text)                # collapse repeated punctuation
    text = re.sub(r"([a-zA-Z])\1{2,}", r"\1\1", text)         # collapse elongated letters
    text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()
    return text

print(clean_text(_demo_raw))

In [ ]:
class TextClsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def make_compute_metrics(average="binary"):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        out = {
            "accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro"),
        }
        if average == "binary":
            out["f1"] = f1_score(labels, preds, average="binary")
            out["precision"] = precision_score(labels, preds, average="binary", zero_division=0)
            out["recall"] = recall_score(labels, preds, average="binary", zero_division=0)
        return out
    return compute_metrics


def make_weighted_trainer_class(class_weights_tensor):
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
            loss = loss_fct(outputs.logits.view(-1, class_weights_tensor.numel()), labels.view(-1))
            return (loss, outputs) if return_outputs else loss
    return WeightedTrainer


def transformer_proba_batch(model, tokenizer, texts, num_labels, batch_size=32):
    # Returns an (N, num_labels) softmax probability matrix.
    model.eval()
    all_proba = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tokenizer(
                texts[i:i + batch_size], truncation=True, padding=True,
                max_length=MAX_LEN, return_tensors="pt"
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            logits = model(**enc).logits
            proba = torch.softmax(logits, dim=-1).cpu().numpy()
            all_proba.append(proba)
    return np.concatenate(all_proba, axis=0)


def build_tfidf_pipeline(C, class_weight="balanced"):
    # word(1-2gram) + char(3-5gram) TF-IDF union, feeding a calibrated LinearSVC.
    word_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=30000, sublinear_tf=True)
    char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=30000, sublinear_tf=True)
    vectorizer = FeatureUnion([("word", word_vec), ("char", char_vec)])
    svm = LinearSVC(C=C, class_weight=class_weight, random_state=SEED)
    clf = CalibratedClassifierCV(svm, method="sigmoid", cv=3)
    return vectorizer, clf


def tune_C(X_all, y_all, skf, candidate_Cs=(0.1, 0.3, 0.5, 1.0, 2.0, 5.0)):
    best_C, best_f1 = None, -1
    for C in candidate_Cs:
        fold_f1s = []
        for tr_idx, te_idx in skf.split(X_all, y_all):
            vec, clf = build_tfidf_pipeline(C)
            Xtr, Xte = vec.fit_transform(X_all[tr_idx]), vec.transform(X_all[te_idx])
            clf.fit(Xtr, y_all[tr_idx])
            fold_f1s.append(f1_score(y_all[te_idx], clf.predict(Xte), average="macro"))
        mean_f1 = np.mean(fold_f1s)
        print(f"  C={C}: CV macro-F1 = {mean_f1:.4f}")
        if mean_f1 > best_f1:
            best_C, best_f1 = C, mean_f1
    print(f"Best C = {best_C} (CV macro-F1 = {best_f1:.4f})")
    return best_C

---
# TASK A — Binary Hate Speech Detection (Hate / Non-Hate)

## A.1 — Load, combine, clean

`binary_train.csv` + `hastika_binary_validation.csv` are concatenated into one labeled pool (the
validation set now has gold labels, so there's no reason to leave that data out of training). A
fresh 85/15 stratified split off this combined pool is used only to tune the ensemble weight and
decision threshold; the **final** models are refit on 100% of the labeled data afterwards.

In [ ]:
bin_train_df = pd.read_csv(BIN_TRAIN_PATH)
bin_val_df = pd.read_csv(BIN_VAL_PATH)
bin_test_df = pd.read_csv(BIN_TEST_PATH)

print("binary_train:", bin_train_df.shape, "| hastika_binary_validation:", bin_val_df.shape, "| hastika_binary_test:", bin_test_df.shape)

LABEL_MAP_A = {"hate": 1, "non-hate": 0, "nonhate": 0, "non_hate": 0, "1": 1, "0": 0}

def normalize_label_a(val):
    s = str(val).strip().lower()
    if s in LABEL_MAP_A:
        return LABEL_MAP_A[s]
    return int(float(s))

bin_all_df = pd.concat([bin_train_df, bin_val_df], ignore_index=True)
bin_all_df["label_int"] = bin_all_df["Label"].apply(normalize_label_a)
bin_all_df["clean_text"] = bin_all_df["Comment"].apply(clean_text)
bin_all_df = bin_all_df[bin_all_df["clean_text"].str.len() > 0].reset_index(drop=True)

bin_test_df["clean_text"] = bin_test_df["Comment"].apply(clean_text)

print("\nCombined labeled Task A pool:", bin_all_df.shape)
print(bin_all_df["label_int"].value_counts())
bin_all_df[["Comment", "clean_text", "label_int"]].head(5)

In [ ]:
bin_tr_split, bin_val_split = train_test_split(
    bin_all_df, test_size=0.15, random_state=SEED, stratify=bin_all_df["label_int"]
)
bin_tr_split = bin_tr_split.reset_index(drop=True)
bin_val_split = bin_val_split.reset_index(drop=True)
y_val_a = bin_val_split["label_int"].values
print("Tuning split:", bin_tr_split.shape, "/", bin_val_split.shape)

## A.2 — TF-IDF(word+char) + Calibrated LinearSVC

In [ ]:
skf_a = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_all_a = bin_all_df["clean_text"].values
y_all_a = bin_all_df["label_int"].values

print("Tuning C for Task A SVM via 5-fold CV on the combined labeled pool:")
best_C_a = tune_C(X_all_a, y_all_a, skf_a)

In [ ]:
tfidf_a, svm_a = build_tfidf_pipeline(best_C_a)
X_tr_tfidf_a = tfidf_a.fit_transform(bin_tr_split["clean_text"])
X_val_tfidf_a = tfidf_a.transform(bin_val_split["clean_text"])
svm_a.fit(X_tr_tfidf_a, bin_tr_split["label_int"].values)

val_proba_svm_a = svm_a.predict_proba(X_val_tfidf_a)[:, 1]
val_preds_svm_a = (val_proba_svm_a >= 0.5).astype(int)
print(classification_report(y_val_a, val_preds_svm_a, target_names=["Non-Hate", "Hate"], digits=4))

## A.3 — MuRIL fine-tuning (class-weighted loss)

In [ ]:
tokenizer_a = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds_a = TextClsDataset(bin_tr_split["clean_text"], bin_tr_split["label_int"], tokenizer_a)
val_ds_a = TextClsDataset(bin_val_split["clean_text"], bin_val_split["label_int"], tokenizer_a)

counts_a = bin_tr_split["label_int"].value_counts().sort_index()
class_weights_a = torch.tensor(
    [len(bin_tr_split) / (2.0 * counts_a[0]), len(bin_tr_split) / (2.0 * counts_a[1])], dtype=torch.float
).to(DEVICE)
print("Class weights [Non-Hate, Hate]:", class_weights_a.tolist())

WeightedTrainerA = make_weighted_trainer_class(class_weights_a)

model_a = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

args_a = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "ckpt_taskA"),
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer_a = WeightedTrainerA(
    model=model_a, args=args_a, train_dataset=train_ds_a, eval_dataset=val_ds_a,
    compute_metrics=make_compute_metrics("binary"),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer_a.train()

In [ ]:
eval_a = trainer_a.evaluate()
print("=== MuRIL (Task A) — Validation Results ===")
for k, v in eval_a.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

val_proba_tf_a = transformer_proba_batch(model_a, tokenizer_a, bin_val_split["clean_text"].tolist(), num_labels=2)[:, 1]
val_preds_tf_a = (val_proba_tf_a >= 0.5).astype(int)
print()
print(classification_report(y_val_a, val_preds_tf_a, target_names=["Non-Hate", "Hate"], digits=4))

## A.4 — Ensemble weight + threshold tuning

In [ ]:
best_w_a, best_t_a, best_f1_a = 0.5, 0.5, -1
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * val_proba_svm_a + (1 - w) * val_proba_tf_a
    for t in np.arange(0.30, 0.71, 0.02):
        f1 = f1_score(y_val_a, (blend >= t).astype(int), average="macro")
        if f1 > best_f1_a:
            best_w_a, best_t_a, best_f1_a = w, t, f1

print(f"Task A — best ensemble weight (SVM share) w={best_w_a:.2f}, threshold t={best_t_a:.2f} -> val macro-F1={best_f1_a:.4f}")

val_proba_ens_a = best_w_a * val_proba_svm_a + (1 - best_w_a) * val_proba_tf_a
val_preds_ens_a = (val_proba_ens_a >= best_t_a).astype(int)

results_a = pd.DataFrame([
    {"model": "TF-IDF+SVM", "f1_macro": f1_score(y_val_a, val_preds_svm_a, average="macro")},
    {"model": "MuRIL",      "f1_macro": f1_score(y_val_a, val_preds_tf_a, average="macro")},
    {"model": "Ensemble",   "f1_macro": best_f1_a},
])
print()
print(results_a)

BEST_MODEL_A = results_a.loc[results_a["f1_macro"].idxmax(), "model"]
print(f"\nSelected for Task A: {BEST_MODEL_A}")

## A.5 — Refit chosen models on ALL labeled Task A data, then predict on the real test set

The weight/threshold/C above were chosen using a held-out split. Now every labeled row (train +
validation combined) is used to fit the final models, since none of it needs to be held back anymore
— the thing we actually need to generalize to is `hastika_binary_test.csv`.

In [ ]:
# Final TF-IDF + SVM, fit on 100% of labeled Task A data
tfidf_a_final, svm_a_final = build_tfidf_pipeline(best_C_a)
X_all_tfidf_a = tfidf_a_final.fit_transform(bin_all_df["clean_text"])
svm_a_final.fit(X_all_tfidf_a, bin_all_df["label_int"].values)

# Final MuRIL, fit on 100% of labeled Task A data (small held-out slice only for early stopping)
final_tr_a, final_es_a = train_test_split(
    bin_all_df, test_size=0.05, random_state=SEED, stratify=bin_all_df["label_int"]
)
train_ds_a_final = TextClsDataset(final_tr_a["clean_text"], final_tr_a["label_int"], tokenizer_a)
es_ds_a_final = TextClsDataset(final_es_a["clean_text"], final_es_a["label_int"], tokenizer_a)

counts_a_f = final_tr_a["label_int"].value_counts().sort_index()
class_weights_a_f = torch.tensor(
    [len(final_tr_a) / (2.0 * counts_a_f[0]), len(final_tr_a) / (2.0 * counts_a_f[1])], dtype=torch.float
).to(DEVICE)
WeightedTrainerAFinal = make_weighted_trainer_class(class_weights_a_f)

model_a_final = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
args_a_final = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "ckpt_taskA_final"),
    num_train_epochs=6, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, warmup_steps=100,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    logging_steps=50, fp16=torch.cuda.is_available(), report_to="none", seed=SEED,
)
trainer_a_final = WeightedTrainerAFinal(
    model=model_a_final, args=args_a_final, train_dataset=train_ds_a_final, eval_dataset=es_ds_a_final,
    compute_metrics=make_compute_metrics("binary"),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer_a_final.train()

In [ ]:
test_proba_svm_a = svm_a_final.predict_proba(tfidf_a_final.transform(bin_test_df["clean_text"]))[:, 1]
test_proba_tf_a = transformer_proba_batch(model_a_final, tokenizer_a, bin_test_df["clean_text"].tolist(), num_labels=2)[:, 1]

if BEST_MODEL_A == "TF-IDF+SVM":
    test_proba_blend_a = test_proba_svm_a
    final_t_a = 0.5
elif BEST_MODEL_A == "MuRIL":
    test_proba_blend_a = test_proba_tf_a
    final_t_a = 0.5
else:
    test_proba_blend_a = best_w_a * test_proba_svm_a + (1 - best_w_a) * test_proba_tf_a
    final_t_a = best_t_a

final_preds_a = (test_proba_blend_a >= final_t_a).astype(int)

submission_a = pd.DataFrame({"id": bin_test_df["id"].values, "label": final_preds_a})
submission_a["label"] = submission_a["label"].map({0: "Non-Hate", 1: "Hate"})

predictions_path_a = os.path.join(OUTPUT_DIR, "predictions_taskA.csv")
submission_a.to_csv(predictions_path_a, index=False, encoding="utf-8")

zip_path_a = os.path.join(OUTPUT_DIR, "predictions_taskA.zip")
with zipfile.ZipFile(zip_path_a, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(predictions_path_a, arcname="predictions.csv")

print(f"Task A model used: {BEST_MODEL_A}")
print(f"Saved: {predictions_path_a}")
print(f"Zipped: {zip_path_a}")
submission_a.head(10)

---
# TASK B — Target Category Classification (6-way)

Same overall approach as Task A (TF-IDF+SVM + MuRIL + tuned ensemble, refit on all labeled data),
adapted for 6 classes: **Gender, Political, Religion, Geo-political, Violence, Others**. The SVM
and the transformer both now output a 6-way probability vector instead of a single Hate probability,
so the ensemble blends two probability matrices and decodes with `argmax` instead of a single
threshold.

## B.1 — Load, combine, clean

In [ ]:
mc_train_df = pd.read_csv(MC_TRAIN_PATH)
mc_val_df = pd.read_csv(MC_VAL_PATH)
mc_test_df = pd.read_csv(MC_TEST_PATH)

print("multiclass_train:", mc_train_df.shape, "| hastika_multiclass_validation:", mc_val_df.shape, "| hastika_multiclass_test:", mc_test_df.shape)

CATEGORIES = ["Gender", "Political", "Religion", "Geo-political", "Violence", "Others"]
CAT2IDX = {c: i for i, c in enumerate(CATEGORIES)}
IDX2CAT = {i: c for c, i in CAT2IDX.items()}

def normalize_category(val):
    s = str(val).strip()
    # tolerate minor case/spacing variants just in case
    for c in CATEGORIES:
        if s.lower().replace("-", "").replace(" ", "") == c.lower().replace("-", "").replace(" ", ""):
            return CAT2IDX[c]
    raise ValueError(f"Unrecognized category: {val!r}")

mc_all_df = pd.concat([mc_train_df, mc_val_df], ignore_index=True)
mc_all_df["label_int"] = mc_all_df["Hate Category"].apply(normalize_category)
mc_all_df["clean_text"] = mc_all_df["Comment"].apply(clean_text)
mc_all_df = mc_all_df[mc_all_df["clean_text"].str.len() > 0].reset_index(drop=True)

mc_test_df["clean_text"] = mc_test_df["Comment"].apply(clean_text)

print("\nCombined labeled Task B pool:", mc_all_df.shape)
print(mc_all_df["Hate Category"].value_counts())
mc_all_df[["Comment", "clean_text", "Hate Category", "label_int"]].head(5)

In [ ]:
mc_tr_split, mc_val_split = train_test_split(
    mc_all_df, test_size=0.15, random_state=SEED, stratify=mc_all_df["label_int"]
)
mc_tr_split = mc_tr_split.reset_index(drop=True)
mc_val_split = mc_val_split.reset_index(drop=True)
y_val_b = mc_val_split["label_int"].values
print("Tuning split:", mc_tr_split.shape, "/", mc_val_split.shape)

## B.2 — TF-IDF(word+char) + Calibrated LinearSVC (multi-class, one-vs-rest)

In [ ]:
skf_b = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_all_b = mc_all_df["clean_text"].values
y_all_b = mc_all_df["label_int"].values

print("Tuning C for Task B SVM via 5-fold CV on the combined labeled pool:")
best_C_b = tune_C(X_all_b, y_all_b, skf_b)

In [ ]:
tfidf_b, svm_b = build_tfidf_pipeline(best_C_b)
X_tr_tfidf_b = tfidf_b.fit_transform(mc_tr_split["clean_text"])
X_val_tfidf_b = tfidf_b.transform(mc_val_split["clean_text"])
svm_b.fit(X_tr_tfidf_b, mc_tr_split["label_int"].values)

val_proba_svm_b = svm_b.predict_proba(X_val_tfidf_b)   # (N, 6)
val_preds_svm_b = np.argmax(val_proba_svm_b, axis=1)
print(classification_report(y_val_b, val_preds_svm_b, target_names=CATEGORIES, digits=4))

## B.3 — MuRIL fine-tuning, 6-way, class-weighted loss

In [ ]:
tokenizer_b = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds_b = TextClsDataset(mc_tr_split["clean_text"], mc_tr_split["label_int"], tokenizer_b)
val_ds_b = TextClsDataset(mc_val_split["clean_text"], mc_val_split["label_int"], tokenizer_b)

counts_b = mc_tr_split["label_int"].value_counts().sort_index()
n_classes_b = len(CATEGORIES)
class_weights_b = torch.tensor(
    [len(mc_tr_split) / (n_classes_b * counts_b.get(i, 1)) for i in range(n_classes_b)], dtype=torch.float
).to(DEVICE)
print("Class weights", dict(zip(CATEGORIES, class_weights_b.tolist())))

WeightedTrainerB = make_weighted_trainer_class(class_weights_b)

model_b = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_classes_b).to(DEVICE)

args_b = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "ckpt_taskB"),
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer_b = WeightedTrainerB(
    model=model_b, args=args_b, train_dataset=train_ds_b, eval_dataset=val_ds_b,
    compute_metrics=make_compute_metrics("macro_only"),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer_b.train()

In [ ]:
eval_b = trainer_b.evaluate()
print("=== MuRIL (Task B) — Validation Results ===")
for k, v in eval_b.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

val_proba_tf_b = transformer_proba_batch(model_b, tokenizer_b, mc_val_split["clean_text"].tolist(), num_labels=n_classes_b)
val_preds_tf_b = np.argmax(val_proba_tf_b, axis=1)
print()
print(classification_report(y_val_b, val_preds_tf_b, target_names=CATEGORIES, digits=4))

## B.4 — Ensemble weight tuning (probability-matrix blend)

In [ ]:
best_w_b, best_f1_b = 0.5, -1
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * val_proba_svm_b + (1 - w) * val_proba_tf_b
    preds = np.argmax(blend, axis=1)
    f1 = f1_score(y_val_b, preds, average="macro")
    if f1 > best_f1_b:
        best_w_b, best_f1_b = w, f1

print(f"Task B — best ensemble weight (SVM share) w={best_w_b:.2f} -> val macro-F1={best_f1_b:.4f}")

val_proba_ens_b = best_w_b * val_proba_svm_b + (1 - best_w_b) * val_proba_tf_b
val_preds_ens_b = np.argmax(val_proba_ens_b, axis=1)

results_b = pd.DataFrame([
    {"model": "TF-IDF+SVM", "f1_macro": f1_score(y_val_b, val_preds_svm_b, average="macro")},
    {"model": "MuRIL",      "f1_macro": f1_score(y_val_b, val_preds_tf_b, average="macro")},
    {"model": "Ensemble",   "f1_macro": best_f1_b},
])
print()
print(results_b)

BEST_MODEL_B = results_b.loc[results_b["f1_macro"].idxmax(), "model"]
print(f"\nSelected for Task B: {BEST_MODEL_B}")

print("\nConfusion matrix (ensemble):")
cm_b = confusion_matrix(y_val_b, val_preds_ens_b)
sns.heatmap(cm_b, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Task B Ensemble — Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## B.5 — Refit chosen models on ALL labeled Task B data, then predict on the real test set

In [ ]:
tfidf_b_final, svm_b_final = build_tfidf_pipeline(best_C_b)
X_all_tfidf_b = tfidf_b_final.fit_transform(mc_all_df["clean_text"])
svm_b_final.fit(X_all_tfidf_b, mc_all_df["label_int"].values)

final_tr_b, final_es_b = train_test_split(
    mc_all_df, test_size=0.08, random_state=SEED, stratify=mc_all_df["label_int"]
)
train_ds_b_final = TextClsDataset(final_tr_b["clean_text"], final_tr_b["label_int"], tokenizer_b)
es_ds_b_final = TextClsDataset(final_es_b["clean_text"], final_es_b["label_int"], tokenizer_b)

counts_b_f = final_tr_b["label_int"].value_counts().sort_index()
class_weights_b_f = torch.tensor(
    [len(final_tr_b) / (n_classes_b * counts_b_f.get(i, 1)) for i in range(n_classes_b)], dtype=torch.float
).to(DEVICE)
WeightedTrainerBFinal = make_weighted_trainer_class(class_weights_b_f)

model_b_final = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_classes_b).to(DEVICE)
args_b_final = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "ckpt_taskB_final"),
    num_train_epochs=8, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, warmup_steps=100,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    logging_steps=50, fp16=torch.cuda.is_available(), report_to="none", seed=SEED,
)
trainer_b_final = WeightedTrainerBFinal(
    model=model_b_final, args=args_b_final, train_dataset=train_ds_b_final, eval_dataset=es_ds_b_final,
    compute_metrics=make_compute_metrics("macro_only"),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer_b_final.train()

In [ ]:
test_proba_svm_b = svm_b_final.predict_proba(tfidf_b_final.transform(mc_test_df["clean_text"]))
test_proba_tf_b = transformer_proba_batch(model_b_final, tokenizer_b, mc_test_df["clean_text"].tolist(), num_labels=n_classes_b)

if BEST_MODEL_B == "TF-IDF+SVM":
    test_proba_blend_b = test_proba_svm_b
elif BEST_MODEL_B == "MuRIL":
    test_proba_blend_b = test_proba_tf_b
else:
    test_proba_blend_b = best_w_b * test_proba_svm_b + (1 - best_w_b) * test_proba_tf_b

final_preds_b = np.argmax(test_proba_blend_b, axis=1)

submission_b = pd.DataFrame({"id": mc_test_df["id"].values, "label": [IDX2CAT[i] for i in final_preds_b]})

predictions_path_b = os.path.join(OUTPUT_DIR, "predictions_taskB.csv")
submission_b.to_csv(predictions_path_b, index=False, encoding="utf-8")

zip_path_b = os.path.join(OUTPUT_DIR, "predictions_taskB.zip")
with zipfile.ZipFile(zip_path_b, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(predictions_path_b, arcname="predictions.csv")

print(f"Task B model used: {BEST_MODEL_B}")
print(f"Saved: {predictions_path_b}")
print(f"Zipped: {zip_path_b}")
submission_b.head(10)

---
## Final — what to submit

In [ ]:
print("TASK A — upload this file to the Task A phase on CodaBench:")
print(" ", zip_path_a, f"(model used: {BEST_MODEL_A}, val macro-F1: {results_a['f1_macro'].max():.4f})")
print()
print("TASK B — upload this file to the Task B phase on CodaBench:")
print(" ", zip_path_b, f"(model used: {BEST_MODEL_B}, val macro-F1: {results_b['f1_macro'].max():.4f})")
print()
print("CodaBench: https://www.codabench.org/competitions/17784/")
print("\nFiles in /kaggle/working/:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", f)

## Notes
1. **Two separate submissions** — `predictions_taskA.zip` goes to Task A's phase, `predictions_taskB.zip` goes to Task B's phase. Each zip contains a single `predictions.csv` (no enclosing folder), matching the official format.
2. **Deadline is 1 Oct** per the earlier binary-task run — confirm Task B's deadline on CodaBench directly, since multi-class phases sometimes close separately.
3. **The mojibake fix is the single biggest change in this notebook.** If your score jumps noticeably versus the earlier binary-only run, that recovered Kannada signal is almost certainly why.
4. If `DATA_DIR` in the CONFIG cell doesn't match your Kaggle dataset's actual mount path, re-check the **"Inspect dataset folder"** cell's output and fix it there before re-running.
5. If you hit a GPU out-of-memory error on either transformer cell, lower `per_device_train_batch_size` from 16 to 8 in that cell's `TrainingArguments`.
6. Task B's **Geo-political** class has the fewest training examples (186) — check its row in the classification report; if its F1 is much lower than the others, that's expected given the data, not a bug.